# Flight Delay Prediction — Training Pipeline

Trains a **Logistic Regression** model to predict whether a US domestic
flight will arrive more than 15 minutes late.

- **Imputation:** KNN Imputer
- **Encoding:** Binary encoding for high-cardinality columns (ORIGIN, DEST) +
  One-Hot encoding for low-cardinality columns (CARRIER, MONTH, DAY_OF_WEEK)
- **Output:** a `flight_delay_model.pkl` file consumed by the Streamlit app
  (`app.py` / `app.ipynb`).

Each step of the original script is defined as a function in its own cell,
then called in order in the "Run the pipeline" section at the bottom —
run the cells top to bottom.


## 1. Imports

- `KNNImputer` — fills missing numeric values using the average of the K
  most similar rows, instead of a single global mean.
- `StandardScaler` — rescales features to mean 0 / std 1, which Logistic
  Regression's gradient descent needs to converge evenly across features.
- `category_encoders` — provides `BinaryEncoder` for high-cardinality
  categorical columns.
- `LogisticRegression` — the model: predicts P(delayed) via a sigmoid
  function applied to a linear combination of features.
- `train_test_split` — splits data into train/test sets.
- `classification_report`, `roc_auc_score`, `confusion_matrix`,
  `ConfusionMatrixDisplay` — evaluation tools.
- `Pipeline` — chains the imputer, scaler and model into one object, so the
  same fitted transformations are applied consistently to train and test
  data (prevents data leakage) and the whole chain can be saved as one file.
- `joblib` — efficient serialization for saving/loading the trained pipeline.
- `matplotlib` (with the `Agg` backend) — used to render and save the
  confusion matrix plot without needing a display.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

from sklearn.preprocessing import StandardScaler
import category_encoders as ce

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.pipeline import Pipeline

import joblib

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 2. Load data

Reads only the columns we need from the raw CSV and renames them to the
naming convention used throughout the rest of the pipeline (matching the
BTS/FAA style column names the app also uses).


In [ ]:
def load_data(filepath):

    raw_cols = [
        'MONTH',               # Month of flight (1-12)
        'DAY_OF_WEEK',         # Day of week (1=Monday, 7=Sunday)
        'AIRLINE',             # Airline carrier code (e.g. 'AA', 'DL')
        'ORIGIN_AIRPORT',      # Origin airport IATA code (e.g. 'JFK')
        'DESTINATION_AIRPORT', # Destination airport IATA code (e.g. 'LAX')
        'SCHEDULED_DEPARTURE', # Scheduled departure time (e.g. 1430 = 2:30 PM)
        'DEPARTURE_DELAY',     # Departure delay in minutes (can be negative = early)
        'TAXI_OUT',            # Taxi-out time in minutes (gate to runway)
        'AIR_TIME',            # Actual airborne time in minutes
        'DISTANCE',            # Distance between airports in miles
        'ARRIVAL_DELAY',       # Arrival delay in minutes - used to build target
        'CANCELLED'            # 1 if flight was cancelled, 0 if not
    ]

    print("Loading dataset...")
    # low_memory=False avoids pandas guessing column dtypes mid-read on a large file
    df = pd.read_csv(filepath, usecols=raw_cols, low_memory=False)

    df = df.rename(columns={
        'AIRLINE': 'OP_UNIQUE_CARRIER',
        'ORIGIN_AIRPORT': 'ORIGIN',
        'DESTINATION_AIRPORT': 'DEST',
        'SCHEDULED_DEPARTURE': 'CRS_DEP_TIME',
        'DEPARTURE_DELAY': 'DEP_DELAY',
        'ARRIVAL_DELAY': 'ARR_DELAY'
    })

    print(f"Raw shape: {df.shape}")
    return df


## 3. Clean data

Removes cancelled flights (they have no arrival delay to predict), drops any
rows with a missing target, and takes a random sample of at most 200,000 rows
to keep training time reasonable.


In [ ]:
def clean_data(df):

    df = df[df['CANCELLED'] == 0].copy()
    df = df.dropna(subset=['ARR_DELAY'])
    df = df.drop(columns=['CANCELLED'])
    df = df.sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)
    print(f"After cleaning: {df.shape}")
    return df


## 4. Feature engineering

- **DEP_HOUR:** `CRS_DEP_TIME` is stored like `1430` (2:30 PM). Integer
  dividing by 100 extracts just the hour (`14`), which captures peak/off-peak
  patterns better than the raw HHMM value.
- **IS_DELAYED:** the binary target. A flight is "delayed" if it arrives more
  than 15 minutes late — the standard FAA/BTS definition.
- `ARR_DELAY` is dropped after creating the target, because keeping it would
  be target leakage (the model would just learn "IS_DELAYED = ARR_DELAY > 15").


In [ ]:
def engineer_features(df):

    df['DEP_HOUR'] = (df['CRS_DEP_TIME'] // 100).astype(int)
    df = df.drop(columns=['CRS_DEP_TIME'])

    # 1 = delayed (arrived >15 min late), 0 = on time or early
    df['IS_DELAYED'] = np.where(df['ARR_DELAY'] > 15, 1, 0)

    df = df.drop(columns=['ARR_DELAY'])

    print(f"Delay rate: {df['IS_DELAYED'].mean():.2%}")  # typically ~18-22%

    return df


## 5. Split features and target

Simple separation of the predictors (`X`) from the label (`y`).


In [ ]:
def split_features_target(df):

    X = df.drop(columns=['IS_DELAYED'])
    y = df['IS_DELAYED']

    return X, y


## 6. Define encoding column groups

- `binary_cols` — high-cardinality categoricals (hundreds of airports), so
  they get binary encoding instead of one-hot to avoid an explosion of columns.
- `ohe_cols` — low-cardinality categoricals, one-hot encoded. MONTH and
  DAY_OF_WEEK are treated as categories (not ordered numbers) here.
- `numeric_cols` — continuous/integer features fed directly into the model.


In [ ]:
def get_column_groups(X):

    binary_cols = ['ORIGIN', 'DEST']

    ohe_cols = ['OP_UNIQUE_CARRIER', 'MONTH', 'DAY_OF_WEEK']

    numeric_cols = ['DEP_DELAY', 'TAXI_OUT', 'AIR_TIME', 'DISTANCE', 'DEP_HOUR']

    return binary_cols, ohe_cols, numeric_cols


## 7. Build the sklearn Pipeline

Chains the KNN imputer, the scaler, and the Logistic Regression model into a
single object, so `.fit()`/`.predict()` runs all three steps in order and the
whole chain can be saved/loaded as one artifact.

- **KNNImputer(n_neighbors=5):** for each missing value, look at the 5 most
  similar rows and fill in their average — smarter than a plain column mean.
- **StandardScaler:** rescales every feature to mean 0 / std 1 so no single
  feature (e.g. DISTANCE, which ranges 0-5000) dominates gradient descent.
- **LogisticRegression:**
  - `max_iter=1000` — enough gradient descent steps to converge on this data
    (the default of 100 often isn't enough).
  - `C=1.0` — regularization strength (inverse); a balanced default.
  - `solver='lbfgs'` — optimizer that works well for medium-sized datasets.
  - `class_weight='balanced'` — automatically up-weights the minority
    "delayed" class (~20% of rows) so the model doesn't just predict
    "on time" for everything.
  - `random_state=42` — reproducible results.


In [ ]:
def build_pipeline(binary_cols, ohe_cols, numeric_cols):

    pipeline = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1000,
            C=1.0,
            solver='lbfgs',
            class_weight='balanced',
            random_state=42
        ))
    ])

    return pipeline


## 8. Preprocess categoricals (before the pipeline)

Categorical encoding is done outside the sklearn `Pipeline` for clarity, and
because `category_encoders` integrates cleanly with pandas DataFrames.

**Binary encoding (ORIGIN, DEST):** each category gets an integer ordinal,
then that integer is represented in binary using separate columns — e.g. with
300 airports we need only `ceil(log2(300)) = 9` columns instead of the 300
one-hot would need.

**One-hot encoding (CARRIER, MONTH, DAY_OF_WEEK):** `pd.get_dummies` creates
one binary column per category. `drop_first=True` removes the first category
to avoid multicollinearity (redundant information).

Encoders/columns are always **fit on train, applied to test** — this ensures
test data never leaks information into the fitting step, and that both sets
end up with identical columns (`reindex` fills anything missing on the test
side with 0).


In [ ]:
def preprocess_categoricals(X_train, X_test, binary_cols, ohe_cols):

    # --- Binary Encoding ---
    binary_encoder = ce.BinaryEncoder(cols=binary_cols, drop_invariant=True)
    X_train = binary_encoder.fit_transform(X_train)  # learn + apply on train
    X_test = binary_encoder.transform(X_test)        # apply the same mapping to test

    # --- One-Hot Encoding ---
    for col in ['MONTH', 'DAY_OF_WEEK']:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=ohe_cols, drop_first=True)

    # Align test columns to train (fills any category missing from test with 0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    print(f"Feature matrix shape after encoding: {X_train.shape}")
    return X_train, X_test, binary_encoder, list(X_train.columns)


## 9. Train and evaluate

Fits the pipeline, predicts on the held-out test set, then reports:

- **Classification report:** precision/recall/F1 per class.
- **AUC-ROC:** the model's ability to rank a delayed flight above an on-time
  one (0.5 = random, 1.0 = perfect).
- **Confusion matrix:** saved as an image (`confusion_matrix.png`) for a
  visual breakdown of true vs. predicted labels.


In [ ]:
def train_and_evaluate(pipeline, X_train, X_test, y_train, y_test):

    print("\nTraining Logistic Regression...")
    pipeline.fit(X_train, y_train)  # runs imputer -> scaler -> model in order

    y_pred = pipeline.predict(X_test)

    # predict_proba returns [[P(on-time), P(delayed)]]; column 1 = P(delayed)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=['On Time', 'Delayed']))

    auc = roc_auc_score(y_test, y_prob)
    print(f"AUC-ROC: {auc:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=['On Time', 'Delayed'])
    disp.plot(ax=ax, colorbar=False)
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.close()
    print("Confusion matrix saved.")

    return y_pred, y_prob, auc


## 10. Save model and metadata

Bundles everything the Streamlit app needs at inference time into a single
`flight_delay_model.pkl` file via `joblib` (preferred over plain `pickle` for
objects containing large numpy arrays, as sklearn models do):

- `pipeline` — the trained KNNImputer + Scaler + LogisticRegression chain
- `binary_encoder` — the fitted BinaryEncoder for ORIGIN/DEST
- `feature_names` — the exact ordered list of columns the model expects
- `ohe_cols` / `binary_cols` — which columns need which encoding


In [ ]:
def save_model(pipeline, binary_encoder, feature_names, ohe_cols, binary_cols):

    joblib.dump({
        'pipeline': pipeline,
        'binary_encoder': binary_encoder,
        'feature_names': feature_names,
        'ohe_cols': ohe_cols,
        'binary_cols': binary_cols
    }, 'flight_delay_model.pkl')

    print("Model saved to flight_delay_model.pkl")


## Run the pipeline

The steps below correspond to the `if __name__ == '__main__':` block in the
original script. In a notebook there's no script-vs-import distinction, so
each step is just run directly, one per cell.

Update `FILEPATH` if `flights.csv` isn't in the same folder as this notebook.
Download it from: https://www.kaggle.com/datasets/usdot/flight-delays


In [ ]:
FILEPATH = 'flights.csv'

df = load_data(FILEPATH)


### Clean the raw data (drop cancellations, missing target, sample to 200k rows)

In [ ]:
df = clean_data(df)


### Engineer features (DEP_HOUR, IS_DELAYED)

In [ ]:
df = engineer_features(df)


### Separate features (X) from target (y)

In [ ]:
X, y = split_features_target(df)


### Identify which columns need which type of encoding

In [ ]:
binary_cols, ohe_cols, numeric_cols = get_column_groups(X)


### Train/test split

80% train, 20% test. `stratify=y` keeps the ~20% delay ratio consistent in
both splits. `random_state=42` makes the split reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


### Encode categorical columns (binary + one-hot)

In [ ]:
X_train_enc, X_test_enc, binary_encoder, feature_names = preprocess_categoricals(
    X_train.copy(), X_test.copy(), binary_cols, ohe_cols
)


### Build the sklearn pipeline (imputer + scaler + model)

In [ ]:
pipeline = build_pipeline(binary_cols, ohe_cols, numeric_cols)


### Train the pipeline and print evaluation metrics

In [ ]:
y_pred, y_prob, auc = train_and_evaluate(
    pipeline, X_train_enc, X_test_enc, y_train, y_test
)


### Save the trained pipeline + encoders for the Streamlit app

In [ ]:
save_model(pipeline, binary_encoder, feature_names, ohe_cols, binary_cols)

print("\nPipeline complete.")
